# Airbnb SQL Analysis (DuckDB in Jupyter)



In [1]:
# Install DuckDB

In [2]:
!pip -q install duckdb


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Imports
# # DuckDB quick sanity check: open an in-memory DB, load the CSV directly with read_csv_auto, and show 5 sample rows.

In [6]:
import duckdb

con = duckdb.connect()

con.execute("""
SELECT *
FROM read_csv_auto('AB_NYC_2019.csv')
LIMIT 5
""").df()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaT,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [29]:
# schema check
# # Show the table schema (column names + data types) for the 'airbnb' table using DuckDB's DESCRIBE.

In [30]:
con.execute("DESCRIBE airbnb;").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,host_id,BIGINT,YES,None,None,None
3,host_name,VARCHAR,YES,None,None,None
4,neighbourhood_group,VARCHAR,YES,None,None,None
5,neighbourhood,VARCHAR,YES,None,None,None
6,latitude,DOUBLE,YES,None,None,None
7,longitude,DOUBLE,YES,None,None,None
8,room_type,VARCHAR,YES,None,None,None
9,price,BIGINT,YES,None,None,None


In [7]:
# Count rows + columns (sanity check)
# # Count total rows in the CSV (quick dataset size check) using DuckDB + read_csv_auto.

In [8]:
con.execute("""
SELECT COUNT(*) AS rows
FROM read_csv_auto('AB_NYC_2019.csv')
""").df()

,rows
0,48895


In [9]:
# Create a VIEW so you don’t repeat read_csv_auto(...)
# Create (or refresh) a DuckDB VIEW named 'airbnb' over the CSV, then count rows from the view to confirm it works.

In [10]:
con.execute("""
CREATE OR REPLACE VIEW airbnb AS
SELECT *
FROM read_csv_auto('AB_NYC_2019.csv');
""")

con.execute("SELECT COUNT(*) AS rows FROM airbnb;").df()

,rows
0,48895


In [11]:
# Query 1: Avg price by neighborhood_group
# # Compute average price and total listings per neighbourhood_group, then sort groups by highest avg price.

In [13]:
con.execute("""
SELECT
  neighbourhood_group,
  ROUND(AVG(price), 2) AS avg_price,
  COUNT(*) AS listings
FROM airbnb
GROUP BY neighbourhood_group
ORDER BY avg_price DESC;
""").df()

,neighbourhood_group,avg_price,listings
0,Manhattan,196.88,21661
1,Brooklyn,124.38,20104
2,Staten Island,114.81,373
3,Queens,99.52,5666
4,Bronx,87.50,1091


In [ ]:
# Query 2: Avg price by neighbourhood_group + room_type
# Break down average price + listing counts by neighbourhood_group AND room_type, sorted within each group by avg price.

In [15]:
con.execute("""
SELECT
  neighbourhood_group,
  room_type,
  ROUND(AVG(price), 2) AS avg_price,
  COUNT(*) AS listings
FROM airbnb
GROUP BY neighbourhood_group, room_type
ORDER BY neighbourhood_group, avg_price DESC;
""").df()

,neighbourhood_group,room_type,avg_price,listings
0,Bronx,Entire home/apt,127.51,379
1,Bronx,Private room,66.79,652
2,Bronx,Shared room,59.80,60
3,Brooklyn,Entire home/apt,178.33,9559
4,Brooklyn,Private room,76.50,10132
5,Brooklyn,Shared room,50.53,413
6,Manhattan,Entire home/apt,249.24,13199
7,Manhattan,Private room,116.78,7982
8,Manhattan,Shared room,88.98,480
9,Queens,Entire home/apt,147.05,2096


In [16]:
# Query 3: Most expensive listings (top 15)
# # List the 15 most expensive listings (with key fields) to spot outliers and sanity-check pricing.

In [17]:
con.execute("""
SELECT
  id,
  name,
  neighbourhood_group,
  neighbourhood,
  room_type,
  price,
  minimum_nights,
  number_of_reviews
FROM airbnb
WHERE price IS NOT NULL
ORDER BY price DESC
LIMIT 15;
""").df()

,id,name,neighbourhood_group,neighbourhood,room_type,price,minimum_nights,number_of_reviews
0,7003697,Furnished room in Astoria apartment,Queens,Astoria,Private room,10000,100,2
1,22436899,1-BR Lincoln Center,Manhattan,Upper West Side,Entire home/apt,10000,30,0
2,13894339,Luxury 1 bedroom apt. -stunning Manhattan views,Brooklyn,Greenpoint,Entire home/apt,10000,5,5
3,9528920,"Quiet, Clean, Lit @ LES & Chinatown",Manhattan,Lower East Side,Private room,9999,99,6
4,31340283,2br - The Heart of NYC: Manhattans Lower East ...,Manhattan,Lower East Side,Entire home/apt,9999,30,0
5,4737930,Spanish Harlem Apt,Manhattan,East Harlem,Entire home/apt,9999,5,1
6,23377410,Beautiful/Spacious 1 bed luxury flat-TriBeCa/Soho,Manhattan,Tribeca,Entire home/apt,8500,30,2
7,2953058,Film Location,Brooklyn,Clinton Hill,Entire home/apt,8000,1,1
8,22779726,East 72nd Townhouse by (Hidden by Airbnb),Manhattan,Upper East Side,Entire home/apt,7703,1,0
9,34895693,Gem of east Flatbush,Brooklyn,East Flatbush,Private room,7500,1,8


In [18]:
# Query 4: “Good deal” (low price + many reviews)
# Find “good value” listings: price ≤ 120 and at least 50 reviews, then rank by most reviews (and cheaper ties first).

In [19]:
con.execute("""
SELECT
  id,
  neighbourhood_group,
  neighbourhood,
  room_type,
  price,
  number_of_reviews
FROM airbnb
WHERE price <= 120
  AND number_of_reviews >= 50
ORDER BY number_of_reviews DESC, price ASC
LIMIT 25;
""").df()

,id,neighbourhood_group,neighbourhood,room_type,price,number_of_reviews
0,9145202,Queens,Jamaica,Private room,47,629
1,903972,Manhattan,Harlem,Private room,49,607
2,903947,Manhattan,Harlem,Private room,49,597
3,891117,Manhattan,Harlem,Private room,49,594
4,10101135,Queens,Jamaica,Private room,47,576
5,8168619,Queens,East Elmhurst,Private room,46,543
6,834190,Manhattan,Lower East Side,Private room,99,540
7,16276632,Queens,East Elmhurst,Private room,48,510
8,166172,Brooklyn,Bushwick,Private room,60,480
9,546383,Queens,Flushing,Private room,55,474


In [20]:
# Query 5: Window function — rank listings by price within each neighbourhood_group
# Use a window function to rank listings by price within each neighbourhood_group, then keep only the top 5 priciest per group.

In [31]:
con.execute("""
SELECT
  id,
  neighbourhood_group,
  neighbourhood,
  room_type,
  price,
  DENSE_RANK() OVER (PARTITION BY neighbourhood_group ORDER BY price DESC) AS price_rank
FROM airbnb
QUALIFY price_rank <= 5
ORDER BY neighbourhood_group, price_rank;
""").df()

,id,neighbourhood_group,neighbourhood,room_type,price,price_rank
0,19698169,Bronx,Riverdale,Private room,2500,1
1,36177241,Bronx,City Island,Entire home/apt,1000,2
2,20330081,Bronx,Riverdale,Shared room,800,3
3,6557289,Bronx,Longwood,Private room,680,4
4,30253236,Bronx,Westchester Square,Entire home/apt,670,5
5,13894339,Brooklyn,Greenpoint,Entire home/apt,10000,1
6,2953058,Brooklyn,Clinton Hill,Entire home/apt,8000,2
7,34895693,Brooklyn,East Flatbush,Private room,7500,3
8,2271504,Brooklyn,Clinton Hill,Entire home/apt,6500,4
9,20654227,Brooklyn,Cypress Hills,Entire home/apt,5000,5


In [22]:
# Query 6: Hosts with many listings (HAVING)
# Identify the busiest hosts: hosts with ≥ 10 listings, showing total listings and their average listing price (top 20 by count).

In [23]:
con.execute("""
SELECT
  host_id,
  host_name,
  COUNT(*) AS total_listings,
  ROUND(AVG(price), 2) AS avg_price
FROM airbnb
GROUP BY host_id, host_name
HAVING COUNT(*) >= 10
ORDER BY total_listings DESC
LIMIT 20;
""").df()

,host_id,host_name,total_listings,avg_price
0,219517861,Sonder (NYC),327,253.20
1,107434423,Blueground,232,303.15
2,30283594,Kara,121,277.53
3,137358866,Kazuya,103,43.83
4,12243051,Sonder,96,213.03
5,16098958,Jeremy & Laura,96,208.96
6,61391963,Corporate Housing,91,146.24
7,22541573,Ken,87,215.44
8,200380610,Pranjal,65,290.23
9,1475015,Mike,52,103.08


In [24]:
# Query 7: Availability buckets
# Bucket listings by availability_365 ranges, then show how many listings fall in each bucket and their average price.

In [25]:
con.execute("""
SELECT
  CASE
    WHEN availability_365 = 0 THEN '0 days'
    WHEN availability_365 BETWEEN 1 AND 60 THEN '1-60'
    WHEN availability_365 BETWEEN 61 AND 180 THEN '61-180'
    ELSE '181-365'
  END AS availability_bucket,
  COUNT(*) AS listings,
  ROUND(AVG(price), 2) AS avg_price
FROM airbnb
GROUP BY availability_bucket
ORDER BY listings DESC;
""").df()

,availability_bucket,listings,avg_price
0,0 days,17533,136.03
1,181-365,14364,178.47
2,61-180,8728,157.46
3,1-60,8270,138.36


In [27]:
# optional thing: Clean averages: filter out extreme/invalid prices (e.g., 0 or very high outliers) so AVG(price) is more realistic.
# Compute average price per neighbourhood_group after filtering out extreme prices (keep only 30–500) to reduce outlier impact.
con.execute("""
SELECT neighbourhood_group, ROUND(AVG(price),2) AS avg_price
FROM airbnb
WHERE price BETWEEN 30 AND 500
GROUP BY neighbourhood_group
ORDER BY avg_price DESC;
""").df()

,neighbourhood_group,avg_price
0,Manhattan,163.57
1,Brooklyn,113.58
2,Queens,93.86
3,Staten Island,92.84
4,Bronx,84.82
